# FIND-Net training on Colab (AAPM CT-MAR)

**Before the first run**
- Runtime: GPU (A100 80 GB).
- Add a Colab secret named `WANDB_API_KEY` (key icon in the left sidebar) and enable notebook access. The key is never written in this notebook.
- Push the repo (incl. `baselines/`, `colab/`, `third_party/findnet/`) to GitHub.

**Order**
- First time: Settings → 1 → 2 → 3 → 4 → 5 → 6 → 7 (paste the reports of 6 and 7 back) → 8.
- **After a disconnect / new runtime:** Settings → 1 → 2 → 3 → 4 → 5 → 8. Training resumes from `CKPT_DIR/last.pt` and continues the same W&B run.

In [1]:
# ---------------- Settings (edit here only) ----------------
import os

PROJECT_ID    = "ctmar-research"
BUCKET        = "gs://ctmar-dataset-uae"
DATA_ROOT     = "/content/data"                                    # runtime LOCAL disk
REPO_URL      = "https://github.com/BARUD77/CT-MAR-Training.git"
REPO_DIR      = "/content/CT-MAR-Training"
CKPT_DIR      = "/content/drive/MyDrive/MAR_project/runs/findnet"  # Google Drive (survives disconnects)
WANDB_PROJECT = "ct-mar"
WANDB_SECRET  = "WANDB_API_KEY"                                    # name of the Colab secret
RUN_NAME      = "gf"                                               # W&B run name -> findnet_gf
CONFIG        = "baselines/findnet/config.yaml"                    # all hyperparameters live here

# zip in the bucket -> (extract dir under DATA_ROOT, top-level folder inside the zip)
DATASET = {
    "MA_image.zip":   ("MA",   "MA_image"),
    "GT.zip":         ("GT",   "GT"),
    "LI.zip":         ("LI",   "LI"),
    "metal_mask.zip": ("mask", "metal_mask"),
}
DIRS = {z: os.path.join(DATA_ROOT, ext, inner) for z, (ext, inner) in DATASET.items()}
MA_DIR, GT_DIR, LI_DIR, MASK_DIR = DIRS["MA_image.zip"], DIRS["GT.zip"], DIRS["LI.zip"], DIRS["metal_mask.zip"]
DATA_ARGS = f'--ma_dir "{MA_DIR}" --gt_dir "{GT_DIR}" --li_dir "{LI_DIR}" --mask_dir "{MASK_DIR}"'
print(DATA_ARGS)
print("CKPT_DIR:", CKPT_DIR)

--ma_dir "/content/data/MA/MA_image" --gt_dir "/content/data/GT/GT" --li_dir "/content/data/LI/LI" --mask_dir "/content/data/mask/metal_mask"
CKPT_DIR: /content/drive/MyDrive/MAR_project/runs/findnet


## 1. Authenticate to Google Cloud

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID}
!gcloud storage ls {BUCKET}

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



## 2. Copy the dataset to the runtime's local disk (skipped if already present)
Downloads only `MA_image`, `GT`, `LI`, `metal_mask` (the test set is not in the bucket). Each zip is downloaded with parallel `gcloud storage cp`, extracted, and deleted right away to keep disk usage low. A `.complete` marker is written after a successful extraction; an interrupted extraction is redone.

In [ ]:
import shutil, subprocess, time

def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def npy_count(d):
    return sum(1 for f in os.listdir(d) if f.endswith(".npy")) if os.path.isdir(d) else 0

zips_dir = os.path.join(DATA_ROOT, "zips")
os.makedirs(zips_dir, exist_ok=True)
for zip_name, (ext, inner) in DATASET.items():
    ext_dir = os.path.join(DATA_ROOT, ext)
    marker = os.path.join(ext_dir, ".complete")
    if os.path.isfile(marker) and npy_count(os.path.join(ext_dir, inner)) > 0:
        print(f"[skip] {zip_name}: already present")
        continue
    shutil.rmtree(ext_dir, ignore_errors=True)  # remove a partial extraction, if any
    t0 = time.time()
    run(f"gcloud storage cp {BUCKET}/{zip_name} {zips_dir}/")
    run(f"unzip -q {zips_dir}/{zip_name} -d {ext_dir}")
    os.remove(os.path.join(zips_dir, zip_name))
    open(marker, "w").close()
    print(f"[done] {zip_name} in {time.time() - t0:.0f}s")

print()
for d in DIRS.values():
    print(f"{d}: {npy_count(d)} .npy files")
total, used, free = shutil.disk_usage("/content")
print(f"Local disk /content: {free / 2**30:.1f} GiB free of {total / 2**30:.1f} GiB")

## 3. Mount Google Drive (checkpoint directory)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoint dir:", CKPT_DIR)
print("contents:", sorted(os.listdir(CKPT_DIR)))

## 4. Get the code and check dependencies
Nothing is installed: FIND-Net only needs torch, numpy, scipy, scikit-learn, PyYAML and wandb, which Colab already has. Do **not** `pip install -r third_party/findnet/requirements.txt` (it pins old torch/numpy/scipy versions).

In [ ]:
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="code commit: %h %ci %s"

assert os.path.isfile("third_party/findnet/Model/findnet.py"), "third_party/findnet missing -- was it pushed?"
assert os.path.isfile("baselines/findnet/train_findnet.py"), "baselines/findnet missing -- was it pushed?"
print(open("third_party/findnet/COMMIT.txt").read())

import importlib, torch
for pkg in ["torch", "numpy", "scipy", "sklearn", "yaml", "wandb"]:
    print(f"{pkg:8s} {getattr(importlib.import_module(pkg), '__version__', '?')}")
print("CUDA", torch.version.cuda, "| GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "NONE")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 5. Log in to Weights & Biases
Reads the API key from the Colab secret `WANDB_API_KEY`; the training process reuses this login.

In [ ]:
import wandb

try:  # Colab secret (browser Colab)
    from google.colab import userdata
    _key = userdata.get(WANDB_SECRET)
except Exception as e:  # e.g. secrets unavailable from the VS Code Colab extension
    from getpass import getpass
    print(f"Colab secret not available ({type(e).__name__}); paste the key (hidden input):")
    _key = getpass("W&B API key: ")
wandb.login(key=_key, relogin=True)
del _key

## 6. Check: shared data split + metrics vs the original trainer code (one-off)
Paste everything from `VERIFY_COMMON REPORT` to `OVERALL` back. Add `--max_val_images 200` if it is slow.

In [ ]:
!python colab/verify_common.py {DATA_ARGS} --max_val_images 0

## 7. Check: FIND-Net forward / GPU memory / overfit / resume / timing (one-off)
Writes only to `/content/findnet_checks` (local disk), never to `CKPT_DIR`; W&B runs are offline. The memory check measures the per-image training memory at 512×512 and runs the later checks with a batch that fits. Paste everything from `FINDNET_CHECKS REPORT` to `OVERALL` back — it contains s/iter, peak GPU memory, the largest batch that fits and the estimated total training time.

In [ ]:
!python colab/findnet_checks.py --config {CONFIG} {DATA_ARGS} --work_dir /content/findnet_checks --wandb_mode offline

## 8. Full training (automatic resume)
⚠️ **Do not start this until the batch-size / image-size decision is made from the cell 7 report.** Measured locally, a training step needs ~18 GiB per 512×512 image, so the paper's batch 6 (~110 GiB) does not fit on an 80 GB A100 with the current `config.yaml`.

**You can re-run this cell after a disconnect** (after Settings and cells 1–5): it loads `CKPT_DIR/last.pt` (model, optimizer, scheduler, RNG, epoch and position inside the epoch, best SSIM) and continues the same W&B run. `last.pt` is saved every epoch and every 1000 iterations; `best.pt` whenever validation masked SSIM improves.

Hyperparameters are in `baselines/findnet/config.yaml`. Only add `--no_resume` to deliberately start over (it overwrites `last.pt`).

In [ ]:
!python baselines/findnet/train_findnet.py --config {CONFIG} {DATA_ARGS} \
    --ckpt_dir "{CKPT_DIR}" --project {WANDB_PROJECT} --run_name {RUN_NAME}

## 9. (Optional) Checkpoint status
Quick look at where training is, e.g. after a disconnect.

In [ ]:
import time, torch
for name in ["last.pt", "best.pt"]:
    p = os.path.join(CKPT_DIR, name)
    if not os.path.isfile(p):
        print(f"{name}: not found")
        continue
    ck = torch.load(p, map_location="cpu", weights_only=False)
    ex = ck["extra"]
    print(f"{name}: global_step={ck['global_step']} | next epoch {ex['next_epoch']} iter {ex['next_iter']}"
          f"/{ex['steps_per_epoch']} | best val SSIM={ck['best_metric']} | W&B run id={ck['wandb_run_id']} | "
          f"modified {time.ctime(os.path.getmtime(p))}")